# Kết quả cuối E0–E4 — Phan Trọng Hoàn

E1/E2/E3 và E4 đủ 300 bước. Notebook này đọc evidence thật đã lưu, không tải model hoặc train; chạy được CPU tại repo. Không coi các cell chưa thực thi trong notebook này là lượt suy luận mới. Pipeline huấn luyện/suy luận: src/run.py, src/train.py, src/inference.py; chỉ chạy model trên Colab GPU. Nguồn Colab do sinh viên cung cấp: extended300_results (1).zip, extended300_figures.zip và notebook tải về (1)/(2), giữ riêng local. Xem REPORT.md để biết resume, mất backup E4 v1 và chạy lại v2, AMP scale khác và model_revision=null.

Sau khi Run all thành công, Save a copy in Drive và tải .ipynb để lưu output thật. Không sửa JSON evidence. Biểu đồ F1/CI95 nằm ngay trong output notebook, không sinh file phụ.


## 0. Môi trường và phạm vi
Chọn Colab CPU (Hardware accelerator: None), rồi Run all. Cell đầu chỉ clone source/evidence nếu chưa có workspace; không cài thư viện hoặc tải model/dataset. Chạy local từ thư mục gốc repo. Kết quả train đã có; các output mới là đọc artifact và vẽ hình bằng CPU.


In [ ]:
from pathlib import Path
import json
import os
import sys
import importlib.util
import subprocess
import tempfile

ROOT = Path.cwd()
try:
    in_colab = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    in_colab = False
if in_colab:
    # Fresh source/evidence avoids accidentally reading an old training workspace.
    # No dataset, model or checkpoint downloads.
    ROOT = Path(tempfile.mkdtemp(prefix='day10_submission_', dir='/content'))
    subprocess.run(['git', 'clone', '--depth', '1',
        'https://github.com/naoh-pt/K4-Track-4-Day10-VisionLanguageModel.git', str(ROOT)], check=True)
    os.chdir(ROOT)
    sys.path.insert(0, str(ROOT))
assert (ROOT / 'report/evidence/e3.json').exists(), 'Local: mở từ thư mục gốc repo.'
print('Source:', ROOT)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


## 1. Dữ liệu và split
Không tải lại dữ liệu để xem báo cáo. Meta dưới do prepare_data.py gốc sinh, version d10-21e4070301.


In [ ]:
meta = json.loads((ROOT / 'report/analysis/dataset_meta.json').read_text(encoding='utf-8'))
assert meta['dataset_version'] == 'd10-21e4070301'
print(meta)


## 2. Kiến trúc thực đo
Số tham số được đếm từ base model thật trên Colab trước khi gắn LoRA; không nạp model ở cell này.


In [ ]:
print(json.loads((ROOT / 'report/analysis/architecture.json').read_text()))


## 3. E0–E3: evidence test
JSON gốc scorer, đủ 311 ảnh. E1/E2/E3 đủ 300 optimizer steps. Không tái sinh output từ model.


In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd()
assert (ROOT / 'report/evidence/e3.json').exists(), 'Chạy từ thư mục gốc repo.'
names = ['e0', 'e1', 'e2', 'e3', 'e4_data_size_0.1', 'e4_data_size_1.0']
evidence = {name: json.loads((ROOT / 'report/evidence' / f'{name}.json').read_text(encoding='utf-8')) for name in names}
for name, d in evidence.items():
    assert d['n_images'] == 311 and d['missing_predictions'] == 0
    print(name, 'F1=', d['micro']['f1'], 'CI95=', d['f1_ci95'], 'valid=', d['parse_valid_rate'])


In [ ]:
from src.results import plot_f1_ci
import matplotlib.pyplot as plt
fig = plot_f1_ci(ROOT / 'report/evidence')
plt.show()


## 4. Chọn nền E4 bằng val
Chốt E3 trước test E4; không chọn bằng test. Run10% khởi tạo từ model gốc.


In [ ]:
from src.results import choose_by_val
base = choose_by_val(ROOT / 'report/validation')
assert base == 'e3'
for name in ('e3', 'e4_data_size_0.1', 'e4_data_size_1.0'):
    assert evidence[name]['meta']['actual_steps'] == 300
delta = evidence['e4_data_size_1.0']['micro']['f1'] - evidence['e4_data_size_0.1']['micro']['f1']
print('F1 giảm khi dùng10%:', delta)
print(json.loads((ROOT / 'report/analysis/architecture.json').read_text()))


## 5. Bẫy tọa độ
Cùng prediction E3 300 bước, chấm norm1000 và cố ý chấm sai như pixel; JSON hợp lệ không bảo đảm box đúng.


In [ ]:
coord = json.loads((ROOT / 'report/analysis/coord_scale_check.json').read_text(encoding='utf-8'))
assert abs(coord['correct_norm1000_f1'] - evidence['e3']['micro']['f1']) < 1e-12
print(coord)


## 6. Sáu ảnh minh họa E3 300 bước
Ba mẫu khớp đủ box, ba mẫu FP/FN; GT xanh, prediction đỏ. Đây là ví dụ chọn theo manifest scorer, không phải tỷ lệ lỗi toàn tập.


In [ ]:
try:
    from IPython.display import display, Image
except ImportError:
    # Terminal CPU check: verify files without installing notebook dependencies.
    from PIL import Image as PILImage
    def Image(filename):
        with PILImage.open(filename) as image:
            image.verify()
        return filename
    def display(value):
        print('Image verified:', value)
manifest = json.loads((ROOT / 'report/analysis/figure_manifest.json').read_text(encoding='utf-8'))
for example in manifest['examples']:
    print(example['image_id'], example['caption'])
    display(Image(filename=str(ROOT / example['figure'])))
coord = json.loads((ROOT / 'report/analysis/coord_scale_check.json').read_text(encoding='utf-8'))
assert abs(coord['correct_norm1000_f1'] - evidence['e3']['micro']['f1']) < 1e-12
print(coord)


## 7. E4: thay đổi lượng dữ liệu
10% gồm 142 ảnh so 100% gồm 1422 ảnh; cùng300 bước/horizon 300 và setup E3. Đối chứng100% tái sử dụng prediction E3.


In [ ]:
for name in ('e4_data_size_0.1', 'e4_data_size_1.0'):
    assert evidence[name]['meta']['actual_steps'] == 300
    print(name, evidence[name]['micro'])
print('F1 giảm:', evidence['e4_data_size_1.0']['micro']['f1'] - evidence['e4_data_size_0.1']['micro']['f1'])


## 8. Kiểm tra và giới hạn
Một seed, AMP scale E1/E2 khác E3/E4, thiếu model revision, CI bootstrap theo ảnh. E4 v1 mất ZIP Drive nên đã train lại v2. Báo cáo200 còn trong commit 9e211e9. E5 chưa làm; không coi output trống ở notebook trình bày này là lượt suy luận mới.


In [ ]:
print(json.loads((ROOT / 'report/analysis/verification.json').read_text(encoding='utf-8')))
